# Irodori-TTS VoiceDesign Lab

Irodori-TTS の VoiceDesign checkpoint を使って、同じ自己紹介文を複数の話し方で生成するノートブックです。

最初のモデルロードと Hugging Face からのダウンロードは時間がかかります。2 回目以降はキャッシュされます。

In [1]:
from pathlib import Path
import shutil
import subprocess
import sys
from IPython.display import Audio, display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
OUT_DIR = ROOT / "outputs" / "voice_design_lab"
OUT_DIR.mkdir(parents=True, exist_ok=True)

UV = shutil.which("uv")
if UV is None:
    raise RuntimeError("uv が見つかりません。ターミナルで `which uv` を確認してください。")

def run_in_project(args, *, timeout=None):
    command = [UV, *args]
    print("$", " ".join(command))
    return subprocess.run(
        command,
        cwd=ROOT,
        check=True,
        text=True,
        capture_output=False,
        timeout=timeout,
    )

print("root:", ROOT)
print("notebook python:", sys.executable)
print("uv:", UV)
print("outputs:", OUT_DIR)
run_in_project(["run", "python", "-c", "import sys, torch; from irodori_tts.inference_runtime import default_runtime_device; print('project python:', sys.executable); print('torch:', torch.__version__); print('device:', default_runtime_device())"])


root: /Users/kakuayato/my-company/irodori-tts-lab
notebook python: /Users/kakuayato/anaconda3/envs/al_yolov8/bin/python
uv: /Users/kakuayato/.local/bin/uv
outputs: /Users/kakuayato/my-company/irodori-tts-lab/outputs/voice_design_lab
$ /Users/kakuayato/.local/bin/uv run python -c import sys, torch; from irodori_tts.inference_runtime import default_runtime_device; print('project python:', sys.executable); print('torch:', torch.__version__); print('device:', default_runtime_device())
project python: /Users/kakuayato/my-company/irodori-tts-lab/.venv/bin/python3
torch: 2.10.0
device: mps


CompletedProcess(args=['/Users/kakuayato/.local/bin/uv', 'run', 'python', '-c', "import sys, torch; from irodori_tts.inference_runtime import default_runtime_device; print('project python:', sys.executable); print('torch:', torch.__version__); print('device:', default_runtime_device())"], returncode=0)

In [2]:
# このノートブックは kernel 環境に依存しないよう、推論は常に `uv run python infer.py` で実行します。
# 初回実行時は Hugging Face から model.safetensors と codec weights をダウンロードします。
HF_CHECKPOINT = "Aratako/Irodori-TTS-500M-v2-VoiceDesign"
print("checkpoint:", HF_CHECKPOINT)
print("inference entrypoint:", ROOT / "infer.py")


checkpoint: Aratako/Irodori-TTS-500M-v2-VoiceDesign
inference entrypoint: /Users/kakuayato/my-company/irodori-tts-lab/infer.py


In [3]:
TEXT = """
はじめまして。私はゆんです。研究と開発を通じて、人の思考を助ける道具を作っています。今日は、音声合成の表現力を確かめるために、同じ自己紹介をいくつかの話し方で読み上げています。
""".strip()

STYLE_PRESETS = {
    "confident_clear": "明るく自信に満ちた若い女性の声。背筋を伸ばして、ハキハキと、語尾まで明瞭に、少し速めのテンポで堂々と自己紹介してください。",
    "confident_calm": "落ち着いた自信のある女性の声。低すぎない自然な声で、聞き手に安心感を与えるように、ゆっくり丁寧に、しかし迷いなく話してください。",
    "nervous_mumbling": "若い女性の声。とても自信がなく不安そう。声量は小さく、息混じりで弱々しい。ゆっくり、短い間を多めに入れ、語尾は少し消え入りそうに下げてください。堂々とせず、視線を落としてぼそぼそ話すように自己紹介してください。",
    "shy_soft": "内気でやわらかい女性の声。距離感は近く、声量は控えめで、少し照れながら、優しく自然に話してください。",
}

TEXT


'はじめまして。私はゆんです。研究と開発を通じて、人の思考を助ける道具を作っています。今日は、音声合成の表現力を確かめるために、同じ自己紹介をいくつかの話し方で読み上げています。'

In [4]:
def synthesize_style(
    name: str,
    caption: str,
    *,
    seed: int = 42,
    num_steps: int = 24,
    cfg_scale_text: float = 3.0,
    cfg_scale_caption: float = 3.5,
) -> Path:
    out_path = OUT_DIR / f"{name}_seed{seed}_steps{num_steps}.wav"
    args = [
        "run",
        "python",
        "infer.py",
        "--hf-checkpoint",
        HF_CHECKPOINT,
        "--text",
        TEXT,
        "--caption",
        caption,
        "--no-ref",
        "--num-steps",
        str(num_steps),
        "--seed",
        str(seed),
        "--cfg-scale-text",
        str(cfg_scale_text),
        "--cfg-scale-caption",
        str(cfg_scale_caption),
        "--output-wav",
        str(out_path),
    ]
    run_in_project(args, timeout=900)
    print("saved:", out_path)
    return out_path


In [5]:
# まずは狙いの 2 種類だけ生成します。
paths = []
for name in ["confident_clear", "nervous_mumbling"]:
    print("\n===", name, "===")
    paths.append(synthesize_style(name, STYLE_PRESETS[name], seed=20260505, num_steps=24))

for path in paths:
    print(path.name)
    display(Audio(filename=str(path)))



=== confident_clear ===
$ /Users/kakuayato/.local/bin/uv run python infer.py --hf-checkpoint Aratako/Irodori-TTS-500M-v2-VoiceDesign --text はじめまして。私はゆんです。研究と開発を通じて、人の思考を助ける道具を作っています。今日は、音声合成の表現力を確かめるために、同じ自己紹介をいくつかの話し方で読み上げています。 --caption 明るく自信に満ちた若い女性の声。背筋を伸ばして、ハキハキと、語尾まで明瞭に、少し速めのテンポで堂々と自己紹介してください。 --no-ref --num-steps 24 --seed 20260505 --cfg-scale-text 3.0 --cfg-scale-caption 3.5 --output-wav /Users/kakuayato/my-company/irodori-tts-lab/outputs/voice_design_lab/confident_clear_seed20260505_steps24.wav
[checkpoint] downloaded model.safetensors from hf://Aratako/Irodori-TTS-500M-v2-VoiceDesign -> /Users/kakuayato/.cache/huggingface/hub/models--Aratako--Irodori-TTS-500M-v2-VoiceDesign/snapshots/456e55708e7183f5c7faa1448209d54aa8991451/model.safetensors
[codec] dacvae: hf://Aratako/Semantic-DACVAE-Japanese-32dim -> /Users/kakuayato/.cache/huggingface/hub/models--Aratako--Semantic-DACVAE-Japanese-32dim/snapshots/47376ee24834d7a05a48ebabfe3cde29b3c5e214/weights.pth


/Users/kakuayato/my-company/irodori-tts-lab/.venv/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


info: speaker conditioning is disabled for this checkpoint; ignoring cfg_scale_speaker.
[seed] used_seed: 20260505
Saved: /Users/kakuayato/my-company/irodori-tts-lab/outputs/voice_design_lab/confident_clear_seed20260505_steps24.wav
[timing] ---- post-model-load to decode ----
[timing] tokenize_text: 0.7 ms
[timing] prepare_reference: 0.0 ms
[timing] sample_rf: 7453.5 ms
[timing] unpatchify_latent: 0.0 ms
[timing] decode_latent: 4229.0 ms
[timing] total_to_decode: 11.687 s
saved: /Users/kakuayato/my-company/irodori-tts-lab/outputs/voice_design_lab/confident_clear_seed20260505_steps24.wav

=== nervous_mumbling ===
$ /Users/kakuayato/.local/bin/uv run python infer.py --hf-checkpoint Aratako/Irodori-TTS-500M-v2-VoiceDesign --text はじめまして。私はゆんです。研究と開発を通じて、人の思考を助ける道具を作っています。今日は、音声合成の表現力を確かめるために、同じ自己紹介をいくつかの話し方で読み上げています。 --caption 若い女性の声。とても自信がなく不安そう。声量は小さく、息混じりで弱々しい。ゆっくり、短い間を多めに入れ、語尾は少し消え入りそうに下げてください。堂々とせず、視線を落としてぼそぼそ話すように自己紹介してください。 --no-ref --num-steps 24 --seed 20260505 --cfg-scale-text 3.

/Users/kakuayato/my-company/irodori-tts-lab/.venv/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


info: speaker conditioning is disabled for this checkpoint; ignoring cfg_scale_speaker.
[seed] used_seed: 20260505
Saved: /Users/kakuayato/my-company/irodori-tts-lab/outputs/voice_design_lab/nervous_mumbling_seed20260505_steps24.wav
[timing] ---- post-model-load to decode ----
[timing] tokenize_text: 0.8 ms
[timing] prepare_reference: 0.0 ms
[timing] sample_rf: 7164.3 ms
[timing] unpatchify_latent: 0.0 ms
[timing] decode_latent: 4281.0 ms
[timing] total_to_decode: 11.448 s
saved: /Users/kakuayato/my-company/irodori-tts-lab/outputs/voice_design_lab/nervous_mumbling_seed20260505_steps24.wav
confident_clear_seed20260505_steps24.wav


nervous_mumbling_seed20260505_steps24.wav


In [6]:
# 追加比較。必要なものだけコメントアウトを外してください。
# extra_paths = []
# for name, caption in STYLE_PRESETS.items():
#     print("\n===", name, "===")
#     extra_paths.append(synthesize_style(name, caption, seed=20260505, num_steps=32))
#
# for path in extra_paths:
#     print(path.name)
#     display(Audio(filename=str(path)))


## 調整のコツ

- 話し方が弱いときは `cfg_scale_caption` を `4.0` から `5.0` くらいに上げる。
- 音が不安定なときは `num_steps` を `32` から `40` に上げる。
- 同じ caption で別の声にしたいときは `seed` を変える。
- まず素早く試すときは `num_steps=12`、本番候補は `num_steps=32` 以上がおすすめ。